In [ ]:
print("Hello")

Hello


In [ ]:
# ⚙️ 1. Könyvtárak telepítése
!pip install torch torchvision Pillow tqdm --quiet

In [ ]:
import difflib

# 📜 3. Kategóriák betöltése
cat_url = 'https://raw.githubusercontent.com/csailvision/places365/master/categories_places365.txt'
categories = requests.get(cat_url).text.strip().split('\n')
categories = [line.split(' ')[0][3:] for line in categories]

# ✅ 3.1. Csak a megadott 121 kategória elfogadása
valid_categories = set('''
stage
sky
orchestra_pit
arena
music_studio
airfield
florist_shop
field_road
discotheque
heliport
botanical_garden
canal
lake
forest
wheat_field
orchard
television_studio
nursing_home
veterinarians_office
park
village
church
promenade
pub
beauty_salon
catacomb
vegetable_garden
runway
cockpit
ballroom
hangar
forest_path
beer_garden
bridge
basement
harbor
field
science_museum
ice_floe
junkyard
rainforest
corn_field
aquarium
fountain
elevator
museum
lagoon
movie_theater
campus
pier
plaza
bamboo_forest
elevator_shaft
tree_farm
zen_garden
classroom
archive
marsh
amphitheater
natural_history_museum
auditorium
server_room
beer_hall
picnic_area
army_base
bakery
beach
jewelry_shop
roof_garden
amusement_park
tundra
booth
highway
mosque
fishpond
art_school
dressing_room
home_theater
hospital_room
bar
parking_lot
conference_center
ice_shelf
railroad_track
vineyard
street
cemetery
delicatessen
tree_house
landfill
sandbox
repair_shop
swamp
creek
forest_road
islet
raft
tower
alley
driveway
auto_factory
engine_room
kennel
art_gallery
ice_cream_parlor
pond
desert
lecture_room
banquet_hall
playground
watering_hole
hayfield
medina
snowfield
physics_laboratory
berth
water_tower
dorm_room
palace
bazaar
pet_shop
'''.strip().split('\n'))

import difflib

# 📜 Teljes Places365 kategóriaadat (név + indoor/outdoor)
cat_url = 'https://raw.githubusercontent.com/csailvision/places365/master/categories_places365.txt'
categories_raw = requests.get(cat_url).text.strip().split('\n')
full_categories = []
for line in categories_raw:
    parts = line.strip().split(' ')
    cat_name = parts[0][3:]
    cat_type = parts[1] if len(parts) > 1 else 'unknown'
    full_categories.append((cat_name, cat_type))

official_names = set(name for name, _ in full_categories)
not_found = sorted(valid_categories - official_names)

print(f"\n❌ Nem található kategóriák a hivatalos listában ({len(not_found)}):\n")

# 🔍 Helyettesítési javaslatok
for cat in not_found:
    suggestions = difflib.get_close_matches(cat, official_names, n=1, cutoff=0.5)
    if suggestions:
        best = suggestions[0]
        best_type = next(t for n, t in full_categories if n == best)
        print(f"  - '{cat}' → '{best}'   # típus: {best_type}")
    else:
        print(f"  - '{cat}' → ⚠️ Nincs javaslat")



❌ Nem található kategóriák a hivatalos listában (19):

  - 'arena' → 'barn'   # típus: 40
  - 'bakery' → 'bakery/shop'   # típus: 31
  - 'bazaar' → 'bar'   # típus: 39
  - 'booth' → 'phone_booth'   # típus: 263
  - 'canal' → 'canal/urban'   # típus: 79
  - 'church' → 'church/indoor'   # típus: 90
  - 'desert' → 'desert_road'   # típus: 118
  - 'elevator' → 'elevator/door'   # típus: 129
  - 'field' → 'hayfield'   # típus: 173
  - 'florist_shop' → 'florist_shop/indoor'   # típus: 147
  - 'forest' → 'rainforest'   # típus: 279
  - 'hangar' → 'hangar/indoor'   # típus: 169
  - 'kennel' → 'kennel/outdoor'   # típus: 201
  - 'lake' → 'palace'   # típus: 252
  - 'mosque' → 'mosque/outdoor'   # típus: 230
  - 'movie_theater' → 'home_theater'   # típus: 177
  - 'museum' → 'mausoleum'   # típus: 226
  - 'pub' → ⚠️ Nincs javaslat
  - 'stage' → 'stable'   # típus: 311


In [ ]:
# ⚙️ Könyvtárak telepítése
!pip install torch torchvision Pillow tqdm --quiet

import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os, shutil, zipfile
from tqdm import tqdm
import requests

# ⚡ 1. GPU detektálás
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU használatban: {device}')

# 🧠 2. Modell betöltés (Places365 ResNet50)
model_url = 'http://places2.csail.mit.edu/models_places365/resnet50_places365.pth.tar'
model_path = 'resnet50_places365.pth.tar'

if not os.path.exists(model_path):
    print("Modell letöltése...")
    !wget -q $model_url -O $model_path

model = models.resnet50(num_classes=365)
checkpoint = torch.load(model_path, map_location=device)
state_dict = {k.replace('module.', ''): v for k, v in checkpoint['state_dict'].items()}
model.load_state_dict(state_dict)
model.eval().to(device)
print("Modell betöltve.")

# 📜 3. Kategóriák betöltése
cat_url = 'https://raw.githubusercontent.com/csailvision/places365/master/categories_places365.txt'
categories = requests.get(cat_url).text.strip().split('\n')
categories = [line.split(' ')[0][3:] for line in categories]

# ✅ 3.1. Csak a megadott 121 kategória elfogadása
valid_categories = set('''
stage/indoor
sky
orchestra_pit
arena/performance
music_studio
airfield
florist_shop/indoor
field_road
discotheque
heliport
botanical_garden
canal/urban
lake/natural
forest/broadleaf
wheat_field
orchard
television_studio
nursing_home
veterinarians_office
park
village
church/indoor
promenade
pub/indoor
beauty_salon
catacomb
vegetable_garden
runway
cockpit
ballroom
hangar/indoor
forest_path
beer_garden
bridge
basement
harbor
field/cultivated
science_museum
ice_floe
junkyard
rainforest
corn_field
aquarium
fountain
elevator/door
museum/indoor
lagoon
movie_theater/indoor
campus
pier
plaza
bamboo_forest
elevator_shaft
tree_farm
zen_garden
classroom
archive
marsh
amphitheater
natural_history_museum
auditorium
server_room
beer_hall
picnic_area
army_base
bakery/shop
beach
jewelry_shop
roof_garden
amusement_park
tundra
booth/indoor
highway
mosque/outdoor
fishpond
art_school
dressing_room
home_theater
hospital_room
bar
parking_lot
conference_center
ice_shelf
railroad_track
vineyard
street
cemetery
delicatessen
tree_house
landfill
sandbox
repair_shop
swamp
creek
forest_road
islet
raft
tower
alley
driveway
auto_factory
engine_room
kennel/outdoor
art_gallery
ice_cream_parlor
pond
desert/vegetation
lecture_room
banquet_hall
playground
watering_hole
hayfield
medina
snowfield
physics_laboratory
berth
water_tower
dorm_room
palace
bazaar/indoor
pet_shop
'''.strip().split('\n'))

valid_category_indices = [i for i, cat in enumerate(categories) if cat in valid_categories]
print(f"✅ Elfogadott kategóriák száma: {len(valid_category_indices)}")

# 🎯 4. Előfeldolgozás
def transform_image(img):
    if img.size != (224, 224):
        img = img.resize((224, 224))
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return transform(img)

# 🔍 5. Képosztályozás – csak a megadott kategóriák közül választ
def classify_image(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        input_tensor = transform_image(img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(input_tensor)[0]  # shape: (365,)
        filtered_logits = output[valid_category_indices]
        top_idx = filtered_logits.argmax().item()
        real_idx = valid_category_indices[top_idx]
        return categories[real_idx]
    except Exception as e:
        print(f'HIBA: {img_path} – {e}')
        return 'unknown'

# 🗂️ 6. Kicsomagolás (feltöltött ZIP fájl)
uploaded_zip = '/content/unsorted_db.zip'
extract_dir = '/content'

# 📁 Ha már létezik az unsorted_db mappa, töröljük
unsorted_path = os.path.join(extract_dir, 'unsorted_db')
if os.path.exists(unsorted_path):
    print("⚠️ Meglévő 'unsorted_db' mappa törlése...")
    shutil.rmtree(unsorted_path)

# 📦 ZIP kicsomagolása
print("ZIP fájl kicsomagolása...")
with zipfile.ZipFile(uploaded_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("✅ Kicsomagolás kész.")


# 🔎 7. Képek összegyűjtése
valid_exts = ('.jpg', '.jpeg', '.png')
all_files = []
for root, _, files in os.walk(extract_dir):
    for file in files:
        if file.lower().endswith(valid_exts):
            all_files.append(os.path.join(root, file))

print(f'Feldolgozandó képek száma: {len(all_files)}')

# 📋 8. Osztályozás
classified = []
for img_path in tqdm(all_files, desc="📊 Képek osztályozása"):
    category = classify_image(img_path)
    if category != 'unknown':
        classified.append((img_path, category))

# 🧹 9. Kategóriákba rendezés
target_root = '/content/sorted_db'
os.makedirs(target_root, exist_ok=True)

for img_path, category in tqdm(classified, desc="💾 Képek áthelyezése"):
    target_dir = os.path.join(target_root, category)
    os.makedirs(target_dir, exist_ok=True)

    file_name = os.path.basename(img_path)
    new_path = os.path.join(target_dir, file_name)

    try:
        shutil.move(img_path, new_path)
    except Exception as e:
        print(f'Nem sikerült áthelyezni: {img_path} – {e}')

# 📦 10. Újratömörítés
zip_output = '/content/sorted_db.zip'
print("Kategorizált képek tömörítése...")

def zipdir(path, ziph):
    for root, _, files in os.walk(path):
        for file in files:
            ziph.write(os.path.join(root, file),
                       os.path.relpath(os.path.join(root, file), path))

with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(target_root, zipf)

print(f'✅ Tömörített fájl elkészült: {zip_output}')


GPU használatban: cuda
Modell betöltve.
✅ Elfogadott kategóriák száma: 121
⚠️ Meglévő 'unsorted_db' mappa törlése...
ZIP fájl kicsomagolása...
✅ Kicsomagolás kész.
Feldolgozandó képek száma: 25491


💾 Képek áthelyezése: 100%|██████████| 25491/25491 [00:00<00:00, 30030.60it/s]


Kategorizált képek tömörítése...
✅ Tömörített fájl elkészült: /content/sorted_db.zip
